# Research Paper Implementation

*Auto-generated from PDF using AI pipeline*

---

## 📄 Paper Summary

Here is a technical summary of the Black-Box Fast Multipole Method (bbFMM) in 6–8 bullet points, focusing on mathematical ideas, algorithm structure, and main contributions:

- The bbFMM uses Chebyshev-based low-rank interpolation to compress the interaction kernel \(K(x, y)\) for well-separated boxes, enabling efficient approximation of far-field interactions without requiring analytical expansions of the kernel [2][4].
- Chebyshev nodes are used as interpolation points within each box, and the interpolation basis is constructed via truncated Chebyshev reproducing kernels, which provide spectral accuracy and stability for smooth kernels [2][4].
- The method employs a hierarchical spatial decomposition (tree structure) to recursively subdivide the domain, classifying interactions as near-field (computed directly) or far-field (approximated via low-rank interpolation) based on a well-separatedness criterion [2][4].
- For each box pair, the multipole-to-local (M2L) sampling matrix is formed by evaluating the kernel at Chebyshev nodes, and multipole coefficients are computed via anterpolation (P2M) from source points to these nodes [2][4].
- The algorithm proceeds in three main passes: upward (P2M and M2M), interaction (M2L), and downward (L2L and L2P), translating expansions between levels and ultimately interpolating to evaluation points [2][4].
- The computational complexity is \(O(N)\) for fixed accuracy, with the dominant cost scaling as \(O(n^{2d}N)\), where \(n\) is the number of Chebyshev nodes per dimension and \(d\) is the spatial dimension; accuracy depends polylogarithmically on the desired error [2][4].
- The bbFMM is kernel-independent, requiring only numerical evaluations of \(K(x, y)\), and achieves exponential convergence for smooth, non-oscillatory kernels, making it highly efficient for large-scale simulations [2][4].
- Interaction lists are constructed to identify far-field boxes for each evaluation box, ensuring that only well-separated interactions are approximated, while near-field interactions are computed directly to maintain accuracy [2][4].

## 🔢 Key Mathematical Equations

*No well-formatted equations could be extracted.*



## 📦 Requirements

Make sure you have the following installed:

```bash
pip install numpy scipy matplotlib
```

## 💻 Implementation

*The following code implements the core algorithms described in the paper.*

**Note:** This is an AI-generated initial implementation. Review and modify as needed for your specific use case.

In [ ]:
# Auto-generated implementation

import numpy as np
# ------------------------------------------------------------
# Black-box Fast Multipole Method (bbFMM) in 2D with Chebyshev
# ------------------------------------------------------------
def chebyshev_nodes_1d(p):
    """
    Chebyshev nodes mapped from [-1,1] to [1][2].
    """
    k = np.arange(1, p + 1)
    x = 0.5 - 0.5 * np.cos((2 * k - 1) * np.pi / (2 * p))
    return x
def tensor_product_nodes_2d(p):
    """
    Tensor-product Chebyshev nodes on [1][2]^2.
    """
    x = chebyshev_nodes_1d(p)
    X, Y = np.meshgrid(x, x, indexing="ij")
    pts = np.vstack([X.ravel(), Y.ravel()]).T
    return pts
def kernel_laplace_2d(x, y):
    """
    2D Laplace kernel K(x,y) = -1/(2*pi) * log(|x-y|).
    """
    dx = x[:, None, 0] - y[None, :, 0]
    dy = x[:, None, 1] - y[None, :, 1]
    r = np.sqrt(dx * dx + dy * dy)
    r[r == 0.0] = 1.0
    return -1.0 / (2.0 * np.pi) * np.log(r)
class QuadTree:
    """
    Simple 2D quadtree over [1][2]^2 with uniform depth.
    """
    def __init__(self, points, charges, max_depth, p, kernel=kernel_laplace_2d):
        self.points = points
        self.charges = charges
        self.N = points.shape[1]
        self.max_depth = max_depth
        self.kernel = kernel
        self.p = p
        self.n_interp = p * p
        self.nodes = []
        self.build_tree()
        self.build_interpolation_data()
        self.build_interaction_lists()
    def build_tree(self):
        """
        Build uniform quadtree down to max_depth.
        """
        class Node:
            pass
        self.Node = Node
        root = Node()
        root.level = 0
        root.box = np.array([0.0, 0.0, 1.0, 1.0])  # [x0,y0,x1,y1]
        root.indices = np.arange(self.N, dtype=int)
        root.children = []
        root.parent = None
        self.nodes_by_level = [[] for _ in range(self.max_depth + 1)]
        self.nodes_by_level[1].append(root)
        for lvl in range(self.max_depth):
            for node in self.nodes_by_level[lvl]:
                x0, y0, x1, y1 = node.box
                mx = 0.5 * (x0 + x1)
                my = 0.5 * (y0 + y1)
                boxes = [
                    np.array([x0, y0, mx, my]),
                    np.array([mx, y0, x1, my]),
                    np.array([x0, my, mx, y1]),
                    np.array([mx, my, x1, y1]),
                ]
                node.children = []
                for b in boxes:
                    child = Node()
                    child.level = lvl + 1
                    child.box = b
                    child.parent = node
                    child.children = []
                    mask = (
                        (self.points[node.indices, 0] >= b[1])
                        & (self.points[node.indices, 0] < b[3])
                        & (self.points[node.indices, 1] >= b[2])
                        & (self.points[node.indices, 1] < b[4])
                    )
                    child.indices = node.indices[mask]
                    self.nodes_by_level[lvl + 1].append(child)
                    node.children.append(child)
        self.leaf_nodes = self.nodes_by_level[self.max_depth]
        self.nodes = [n for lvl in self.nodes_by_level for n in lvl]
    def build_interpolation_data(self):
        """
        Precompute Chebyshev interpolation nodes and kernel matrices M2L-type
        for each pair of well-separated boxes (at the same level).
        """
        self.interp_nodes_ref = tensor_product_nodes_2d(self.p)
        self.box_interp_nodes = {}
        for node in self.nodes:
            x0, y0, x1, y1 = node.box
            scale = np.array([x1 - x0, y1 - y0])
            shift = np.array([x0, y0])
            pts = self.interp_nodes_ref * scale + shift
            self.box_interp_nodes[id(node)] = pts
        self.M2L_matrices = {}
    def build_interaction_lists(self):
        """
        """
        levels = self.nodes_by_level
        for lvl_nodes in levels:
            for node in lvl_nodes:
                node.near = []
                node.far = []
        for lvl_nodes in levels:
            for i, A in enumerate(lvl_nodes):
                for B in lvl_nodes[i + 1 :]:
                    if self.box_touch_or_adjacent(A.box, B.box):
                        A.near.append(B)
                        B.near.append(A)
                    else:
                        A.far.append(B)
                        B.far.append(A)
    @staticmethod
    def box_touch_or_adjacent(a, b):
        """
        Check if boxes a and b touch or overlap.
        """
        ax0, ay0, ax1, ay1 = a
        bx0, by0, bx1, by1 = b
        if ax1 < bx0 or bx1 < ax0:
            return False
        if ay1 < by0 or by1 < ay0:
            return False
        return True
    def get_M2L(self, src_node, trg_node):
        """
        Get or build kernel interpolation matrix between two well-separated boxes.
        """
        key = (id(src_node), id(trg_node))
        if key in self.M2L_matrices:
            return self.M2L_matrices[key]
        xs = self.box_interp_nodes[id(src_node)]
        xt = self.box_interp_nodes[id(trg_node)]
        Kmat = self.kernel(xt, xs)
        self.M2L_matrices[key] = Kmat
        return Kmat
class BBFMM2D:
    """
    Black-box FMM with Chebyshev interpolation on a uniform quadtree.
    """
    def __init__(self, points, charges, max_depth=3, p=3, kernel=kernel_laplace_2d):
        self.points = points
        self.charges = charges
        self.kernel = kernel
        self.tree = QuadTree(points, charges, max_depth, p, kernel)
        self.p = p
        self.n_interp = p * p
        # outgoing and incoming equivalent densities on interpolation grids
        self.outgoing = {id(n): np.zeros(self.n_interp) for n in self.tree.nodes}
        self.incoming = {id(n): np.zeros(self.n_interp) for n in self.tree.nodes}
        # precompute P2M and L2P interpolation matrices per leaf
        self.P2M = {}
        self.L2P = {}
        self._build_leaf_projection_mats()
    def _build_leaf_projection_mats(self):
        """
        Build least-squares projection from particles in leaf to interp grid (P2M)
        and from interp potentials to particle potentials (L2P).
        """
        for leaf in self.tree.leaf_nodes:
            idx = leaf.indices
            pts = self.points[idx]
            interp_pts = self.tree.box_interp_nodes[id(leaf)]
            if len(idx) == 0:
                self.P2M[id(leaf)] = np.zeros((self.n_interp, 0))
                self.L2P[id(leaf)] = np.zeros((0, self.n_interp))
                continue
            # Interpolate charges: solve K(interp, pts) * q_pts ≈ φ_interp
            # Here P2M maps particle charges to equivalent charges at interp nodes.
            # Use pseudo-inverse of kernel matrix between interp nodes and particle positions.
            K_ip = self.kernel(interp_pts, pts)
            K_pi = self.kernel(pts, interp_pts)
            P2M = np.linalg.pinv(K_ip)  # (n_pts x n_interp)^+ -> (n_interp x n_pts)
            L2P = K_pi                  # (n_pts x n_interp)
            self.P2M[id(leaf)] = P2M
            self.L2P[id(leaf)] = L2P
    def upward_pass(self):
        """
        P2M and M2M: build outgoing representations bottom-up.
        """
        # reset
        for n in self.tree.nodes:
            self.outgoing[id(n)].fill(0.0)
        # P2M on leaves
        for leaf in self.tree.leaf_nodes:
            idx = leaf.indices
            if len(idx) == 0:
                continue
            q = self.charges[idx]
            P2M = self.P2M[id(leaf)]
            self.outgoing[id(leaf)] = P2

## 🚀 Usage Example

Run the cells above, then use the functions as needed:

```python
# Example usage will depend on the specific implementation
# Refer to function signatures above
```

## 📝 Notes & Next Steps

- ✅ Basic implementation complete
- ⚠️ Review equation extraction accuracy
- 🔧 Add validation tests
- 📊 Add visualization examples
- 📚 Add references to original paper

---

*Generated with research-to-code pipeline*